In [ ]:
from enum import Enum 
import pandas as pd
import itertools

class HeaderEntries(Enum):
    TIMESTAMP = "timestamp"
    MEM_VM = "vm rss (KB)"
    MEM_SERVER = "server hwm (KB)"
    TRACE_GROUP = "trace_group"
    TRACE = "trace_name"
    ALLOCATOR = "allocator"
    SIZE_MULTIPLE = "size_multiple"
    QUERY_COLLECTION = "query_collection"
    PAGE_SIZE = "page_size"
    AMT_PAGES = "amt_pages"
    QUERY = "identifier"

class AllocatorTypes(Enum):
    FFAO = "FFAO"
    NFAO = "NFAO"
    BFAO = "BFAO"

class SizeMultiples(Enum):
    SM1 = 1
    SM16 = 16
    SM64 = 64

class TraceGroups(Enum):
    SIM = "_0_synth_sim"
    RANDOMIZED= "_1_synth_randomized_even_spread"
    STACK = "_2_synth_stack"
    LARSON = "_3_program_larson"
    CFRAC = "_4_program_cfrac"

class Traces(Enum):
    SIM = "_0_synth_sim_0"
    RANDOMIZED = "_0_synth_randomized_even_spread_0"
    STACK = "_0_synth_stack_0"
    LARSON = "_0_larson_0"
    CFRAC = "_0_cfrac_0"

TG_DICT = {
    TraceGroups.SIM.value:Traces.SIM.value,
    TraceGroups.STACK.value:Traces.STACK.value,
    TraceGroups.RANDOMIZED.value:Traces.RANDOMIZED.value,
    TraceGroups.LARSON.value:Traces.LARSON.value,
    TraceGroups.CFRAC.value:Traces.CFRAC.value
}

class Queries(Enum):
    STARTUP_SERVER = "STARTUP_SERVER"
    LIVE_REQ_MEM = "live_requested_memory"
    ADDRESS_START_GEQ = "AddressStartGEQ0" 

class Aggregates(Enum):
    MAX_VM_MEM = "max_vm_mem"
    MAX_SERVER_MEM = "max_server_mem"
    MAX_TOTAL_MEM = "max_total_mem"
    TIME_SPENT = "time_spent"

class MemoryConfig():
    def __init__(self, page_size, amt_pages):
        self.page_size = page_size
        self.amt_pages = amt_pages
    
    def memory_size(self):
        return self.amt_pages * self.page_size

MEMORY_CONFIGS = {
    # 256 KB
    MemoryConfig(1024, 256),
    MemoryConfig(4096, 64),
    MemoryConfig(16384, 16),

    # 320 KB
    MemoryConfig(1024, 320),
    MemoryConfig(4096, 80),
    MemoryConfig(16384, 20),

    # 384 KB
    MemoryConfig(1024, 384),
    MemoryConfig(4096, 96),
    MemoryConfig(16384, 24),
}

TIME_INTERVAL_SAMPLE_SEC = 0.4
COL_SEPARATOR = r" \| "


def get_highest_server_mem(df: pd.DataFrame):
    return df[HeaderEntries.MEM_SERVER.value].max()*1000

def get_highest_vm_mem(df: pd.DataFrame):
    return df[HeaderEntries.MEM_VM.value].max()*1000

def get_highest_total_mem(df: pd.DataFrame):
    return (df[HeaderEntries.MEM_SERVER.value] + df[HeaderEntries.MEM_VM.value]).max()*1000

def get_time_spent(df: pd.DataFrame):
    return (df[HeaderEntries.TIMESTAMP.value].iloc[-1] - df[HeaderEntries.TIMESTAMP.value].iloc[0]).total_seconds()  + TIME_INTERVAL_SAMPLE_SEC

def parse_group(df_filt_mem):
    # Changes all memory to bytes too!!!
    max_vm = get_highest_vm_mem(df_filt_mem)
    max_server = get_highest_server_mem(df_filt_mem)
    max_total = get_highest_total_mem(df_filt_mem)
    time_spent = get_time_spent(df_filt_mem)

    return pd.Series({
        Aggregates.MAX_VM_MEM.value:max_vm,
        Aggregates.MAX_SERVER_MEM.value:max_server,
        Aggregates.MAX_TOTAL_MEM.value:max_total,
        Aggregates.TIME_SPENT.value:time_spent
    })

def generate_mem_df(df_init: pd.DataFrame) -> pd.DataFrame:
    df: pd.DataFrame = df_init.copy()

    df[HeaderEntries.TIMESTAMP.value] = df[HeaderEntries.TIMESTAMP.value].str.replace('_', ' ')
    df[HeaderEntries.TIMESTAMP.value] = pd.to_datetime(df[HeaderEntries.TIMESTAMP.value])

    # String contains an additional space, which is removed here.
    df[HeaderEntries.MEM_SERVER.value] = df[HeaderEntries.MEM_SERVER.value].str.replace(' ', '')
    # N/A only exists in the 'starting server' commands, which are not analyzed. So we replace those values with 0 for easier parsing.
    df[HeaderEntries.MEM_SERVER.value] = df[HeaderEntries.MEM_SERVER.value].str.replace('N/A', '0')
    # Entire column should now be an integer, so parse it as such.
    df[HeaderEntries.MEM_SERVER.value] = df[HeaderEntries.MEM_SERVER.value].astype('Int64')
    
    # Drop query collections, we don't care anyway. There's one collection
    df.drop(HeaderEntries.QUERY_COLLECTION.value, axis=1)
    combinations: list[tuple] = list(itertools.product(
        [m.value for m in TraceGroups],
        [m.value for m in Traces],
        [m.value for m in AllocatorTypes],
        [m.value for m in SizeMultiples],
        [m.value for m in Queries]
    ))
    # Prevent having to iterate over the other filters every time.
    df_filt = df.groupby([HeaderEntries.TRACE_GROUP.value, 
                            HeaderEntries.TRACE.value, 
                            HeaderEntries.ALLOCATOR.value, 
                            HeaderEntries.SIZE_MULTIPLE.value, 
                            HeaderEntries.QUERY.value,
                            HeaderEntries.PAGE_SIZE.value,
                            HeaderEntries.AMT_PAGES.value
                            ])
    return df_filt.apply(parse_group).reset_index()

# Location for the mem track fail you want to parse
mem_track_path = f"../dataset/mem_track_full.txt"

images_folder = "../images"
update_images = False # Set to True if you want images.
df_init: pd.DataFrame = pd.read_csv(mem_track_path, sep=COL_SEPARATOR) # Parse memtrack as csv.
df_mem = generate_mem_df(df_init)

In [ ]:
import plotly.express as px

def make_heap_page_str(df: pd.DataFrame):
      page_mem = (df[HeaderEntries.PAGE_SIZE.value]/1024).astype(int).astype(str) + "KiB"
      total_mem = (df[HeaderEntries.PAGE_SIZE.value]*df[HeaderEntries.AMT_PAGES.value]/1024).astype(int).astype(str) + "KiB"
      return total_mem + "[" + page_mem + "*" + df[HeaderEntries.AMT_PAGES.value].astype(int).astype(str) + "]"

# Saving image to pdf. This was remarkably difficult on linux, so might be platform dependent.
def save_fig_to_pdf(fig, path):

    import plotly.graph_objects as go
    
    import kaleido
    fig.write_image(path, scale=1, width=fig.layout.width, height=fig.layout.height)

df_memtrack = df_mem.copy()
df_memtrack[HeaderEntries.SIZE_MULTIPLE.value] = df_memtrack[HeaderEntries.SIZE_MULTIPLE.value].astype(str)
df_memtrack['HEAP_PAGES'] = df_memtrack[HeaderEntries.PAGE_SIZE.value]* df_memtrack[HeaderEntries.AMT_PAGES.value]
df_memtrack['HEAP_PAGES_STR'] = make_heap_page_str(df_memtrack)

INCLUDED_QUERIES = [
      Queries.LIVE_REQ_MEM.value,
      Queries.ADDRESS_START_GEQ.value
]

# Dictionary for renaming values in the plots.
PLOT_RENAMES = {
    AllocatorTypes.FFAO.value:"FFAO",
    AllocatorTypes.NFAO.value:"NFAO",
    AllocatorTypes.BFAO.value:"BFAO",
    HeaderEntries.SIZE_MULTIPLE.value:"Size Multiple",
    "HEAP_PAGES_STR": "Heap Memory Configuration",
    Aggregates.MAX_SERVER_MEM.value:"Server Mem. (Bytes)",
    Aggregates.MAX_VM_MEM.value:"VM Mem. (Bytes)",
    Queries.LIVE_REQ_MEM.value:"Track Live Requested Memory",
    Queries.ADDRESS_START_GEQ.value:"Check Address Start GEQ 0",
    Aggregates.TIME_SPENT.value:"Time Spent (sec)"
}    

for tg in [m.value for m in TraceGroups]:
    t = TG_DICT[tg]
    df_int = df_memtrack[(df_memtrack[HeaderEntries.TRACE_GROUP.value] == tg) 
                    & (df_memtrack[HeaderEntries.TRACE.value] == t)
                    & (df_mem[HeaderEntries.QUERY.value].isin(INCLUDED_QUERIES))
                ]
    df_sorted = df_int.sort_values(by=['HEAP_PAGES', HeaderEntries.PAGE_SIZE.value], ascending=[True, True])


    fig = px.strip(df_sorted, x='HEAP_PAGES_STR', 
                        y=Aggregates.MAX_SERVER_MEM.value, 
                        facet_row=HeaderEntries.ALLOCATOR.value, 
                        facet_col=HeaderEntries.QUERY.value, 
                        color=HeaderEntries.SIZE_MULTIPLE.value,
                        log_y=True,
                        title="",
                        labels=PLOT_RENAMES
                        )
    
    fig.update_layout(height=500, width=800)
    fig.update_yaxes(
        type="log",
        dtick=1,
        ticks="inside",
        ticklen=8,
        minor=dict(
            dtick="D2",
            ticks="inside",
            ticklen=4,
            showgrid=True
        )
    )

    # Add titles
    fig.for_each_yaxis(lambda y: y.update(title = ''))
    fig.add_annotation(x=-0.05,y=0.5,
        text=Aggregates.MAX_SERVER_MEM.value, textangle=-90,
        xref="paper", yref="paper", font=dict(size=14))
    
    # Cleanup titles
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig.for_each_annotation(lambda a: a.update(text=PLOT_RENAMES.get(a.text)))


    figMaxVM = px.strip(df_sorted, x='HEAP_PAGES_STR', 
            y=Aggregates.MAX_VM_MEM.value, 
            facet_row=HeaderEntries.ALLOCATOR.value, 
            facet_col=HeaderEntries.QUERY.value, 
            color=HeaderEntries.SIZE_MULTIPLE.value,
            labels=PLOT_RENAMES
            )
    figMaxVM.update_layout(height=500, width=1000)
    figMaxVM.for_each_yaxis(lambda y: y.update(title = ''))
    figMaxVM.add_annotation(x=-0.05,y=0.5,
        text=Aggregates.MAX_VM_MEM.value, textangle=-90,
        xref="paper", yref="paper", font=dict(size=14))
    figMaxVM.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    figMaxVM.for_each_annotation(lambda a: a.update(text=PLOT_RENAMES.get(a.text)))

    if update_images:
        save_fig_to_pdf(fig, f"{images_folder}/server_mem_usage/server_mem_usage_{tg}_{t}.pdf")
        save_fig_to_pdf(fig, f"{images_folder}/vm_mem_usage/vm_mem_usage{tg}_{t}.pdf")


for tg in [m.value for m in TraceGroups]:
    t = TG_DICT[tg]
    df_int = df_memtrack[(df_memtrack[HeaderEntries.TRACE_GROUP.value] == tg) 
            & (df_memtrack[HeaderEntries.TRACE.value] == t)
            & (df_mem[HeaderEntries.QUERY.value].isin(INCLUDED_QUERIES))
                ]
    df_sorted = df_int.sort_values(by=['HEAP_PAGES', HeaderEntries.PAGE_SIZE.value], ascending=[True, True])

    fig = px.strip(df_sorted, x='HEAP_PAGES_STR', 
        y=Aggregates.TIME_SPENT.value, 
        facet_row=HeaderEntries.ALLOCATOR.value, 
        facet_col=HeaderEntries.QUERY.value, 
        color=HeaderEntries.SIZE_MULTIPLE.value,
        log_y=True,
        labels=PLOT_RENAMES
    )

    fig.update_yaxes(
        type="log",
        dtick=1,           
        ticks="inside",    
        ticklen=8,
        
        minor=dict(
            dtick="D2", 
            ticks="inside",
            ticklen=4,
            showgrid=True  
        )
    )
    fig.update_layout(height=500, width=800)

    # Add Y-title
    fig.for_each_yaxis(lambda y: y.update(title = ''))
    fig.add_annotation(x=-0.05,y=0.5,
        text=Aggregates.TIME_SPENT.value, textangle=-90,
        xref="paper", yref="paper", font=dict(size=14))
    
    # Cleanup the row/col/other titles, and rename them.
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig.for_each_annotation(lambda a: a.update(text=PLOT_RENAMES.get(a.text)))

    if update_images:
        save_fig_to_pdf(fig, f"{images_folder}/time_spent/time_spent_{tg}_{t}.pdf")




# Prevent filtering of traces which we don't care about.
ACCEPTABLE_TRACES = [
    TraceGroups.CFRAC.value,
    TraceGroups.STACK.value,
    TraceGroups.SIM.value,
    TraceGroups.RANDOMIZED.value,
    TraceGroups.LARSON.value,
]


df_int = df_memtrack[
                (df_mem[HeaderEntries.QUERY.value].isin(INCLUDED_QUERIES))
            ]

df_sorted = df_int.sort_values(by=['HEAP_PAGES', HeaderEntries.PAGE_SIZE.value], ascending=[True, True])

# Generate a graph of a memtrack column, with an overlapping median graph.
def generate_median_overlap_graph(entry, log: bool):
    df_int = df_memtrack[
                    (df_memtrack[HeaderEntries.TRACE_GROUP.value].isin(ACCEPTABLE_TRACES)) 
                    & (df_mem[HeaderEntries.QUERY.value].isin(INCLUDED_QUERIES))
                ]
    df_sorted = df_int.sort_values(by=['HEAP_PAGES', HeaderEntries.PAGE_SIZE.value], ascending=[True, True])

    # Create strip plot for a given value. We use strip to show the differences between each trace. 
    # We don't have alot of traces, so it doesn't take much space for the additional information.
    fig = px.strip(
        df_sorted, x='HEAP_PAGES_STR', 
        y=entry, 
        facet_row=HeaderEntries.ALLOCATOR.value, 
        facet_col=HeaderEntries.QUERY.value, 
        color=HeaderEntries.SIZE_MULTIPLE.value,
        log_y=log,
        title="",
        labels=PLOT_RENAMES)
    fig.update_layout(height=500, width=800)

    # Apply log scaling
    if log:
        fig.update_yaxes(
            type="log",
            dtick=1,
            ticks="inside",
            ticklen=8,
            
            minor=dict(
                dtick="D2",
                ticks="inside",
                ticklen=4,
                showgrid=True
            )
        )

    # Add the Y-axis title
    fig.for_each_yaxis(lambda y: y.update(title = ''))
    fig.add_annotation(x=-0.05,y=0.5,
        text=entry, textangle=-90,
        xref="paper", yref="paper", font=dict(size=14))
    
    # Clean-up the row/col/other titles, and rename them.
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig.for_each_annotation(lambda a: a.update(text=PLOT_RENAMES.get(a.text)))

    # Calculate the median for the trend markers
    df_mean = df_sorted.groupby(['HEAP_PAGES_STR', HeaderEntries.SIZE_MULTIPLE.value, HeaderEntries.ALLOCATOR.value, HeaderEntries.QUERY.value])[entry].median().reset_index()

    # Overlay the bold cluster centers
    figCluster = px.strip(
        df_mean, x='HEAP_PAGES_STR', 
        y=entry, 
        facet_row=HeaderEntries.ALLOCATOR.value, 
        facet_col=HeaderEntries.QUERY.value, 
        color=HeaderEntries.SIZE_MULTIPLE.value,
        log_y=log,
        title="",
        labels=PLOT_RENAMES)
    
    figCluster.update_layout(height=500, width=800)
    figCluster.update_traces(jitter=0.5) # Apply jitter to separate values.

    # Apply log scaling.
    if log:
        figCluster.update_yaxes(
            
            type="log",
            dtick=1,
            ticks="inside",
            ticklen=8,
            
            minor=dict(
                dtick="D2",
                ticks="inside",
                ticklen=4,
                showgrid=True
            )
        )

    # Add Y-title
    figCluster.for_each_yaxis(lambda y: y.update(title = ''))
    figCluster.add_annotation(x=-0.05,y=0.5,
        text=Aggregates.MAX_SERVER_MEM.value, textangle=-90,
        xref="paper", yref="paper", font=dict(size=14))

    # Cleanup the titles and rename    
    figCluster.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    figCluster.for_each_annotation(lambda a: a.update(text=PLOT_RENAMES.get(a.text)))

    # Add diamond symbol for median
    figCluster.update_traces(marker=dict(symbol="diamond", line=dict(width=2, color="black")))

    for trace in figCluster.data:
        fig.add_trace(trace)
    return fig

# Generate the median overlap graphs for server memory, VM memory, and time spent.
fig = generate_median_overlap_graph(Aggregates.MAX_SERVER_MEM.value, True)
fig.show()
if update_images:
    save_fig_to_pdf(fig, f"{images_folder}/server_mem_usage/server_mem_usage_merged.pdf")

fig = generate_median_overlap_graph(Aggregates.MAX_VM_MEM.value, False)
fig.show()
if update_images:
    save_fig_to_pdf(fig, f"{images_folder}/vm_mem_usage/vm_mem_usage_merged.pdf")

fig = generate_median_overlap_graph(Aggregates.TIME_SPENT.value, True)
fig.show()
if update_images:
    save_fig_to_pdf(fig, f"{images_folder}/time_spent/time_spent_merged.pdf")
